# Understand the Gold Benchmark Datasets

This notebook introduces the Gold release: the benchmark-ready layer built on top of Silver. Gold contains a shared benchmark core and four model-specific datasets that all use the same snapshot split and evaluation target.

## Setup

In [ ]:
from pathlib import Path

import pickle

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch
import yaml
from numpy.lib.format import read_array_header_1_0, read_array_header_2_0, read_magic

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

gold_root = Path("../data/gold/lite")
gold_root

## Gold Layout

A Gold tier is organized as one shared `core` plus one directory per model-specific representation. The Lite and Standard tiers have the same structure; Lite is smaller and is the default for this tutorial. To inspect the Standard tier instead, change `gold_root` to `Path("../data/gold/standard")`.

In [ ]:
components = {
    "core": gold_root / "core",
    "tabular": gold_root / "tabular",
    "sequential": gold_root / "sequential",
    "gnn": gold_root / "gnn",
    "graph_event": gold_root / "graph_event",
}

pd.DataFrame(
    [
        {
            "component": name,
            "path": str(path),
            "exists": path.exists(),
        }
        for name, path in components.items()
    ]
)

If the component directories are missing, download Gold Lite from the repository root with:

```bash
python -m scripts.dataset.download.download_dataset --target gold_lite
```

## Helper Functions

The helpers below keep the inspection cells compact. They read Parquet metadata without loading full tables and read NumPy array headers without loading full arrays.

In [ ]:
def read_yaml(path: Path):
    with path.open("r", encoding="utf-8") as f:
        return yaml.safe_load(f) or {}


def require_path(path: Path):
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}. Download the Gold Lite release first.")
    return path


def parquet_info(path: Path):
    parquet_file = pq.ParquetFile(require_path(path))
    return {
        "file": path.name,
        "rows": parquet_file.metadata.num_rows,
        "columns": len(parquet_file.schema.names),
        "first_columns": ", ".join(parquet_file.schema.names[:8]),
    }


def info_table(rows):
    table = pd.DataFrame(rows)
    for column in ["rows", "columns"]:
        if column in table:
            table[column] = table[column].astype("Int64")
    return table


def parquet_head(path: Path, n: int = 5):
    parquet_file = pq.ParquetFile(require_path(path))
    if parquet_file.num_row_groups == 0:
        return pd.DataFrame()
    return parquet_file.read_row_group(0).to_pandas().head(n)


def npy_info(path: Path):
    with require_path(path).open("rb") as f:
        version = read_magic(f)
        if version == (1, 0):
            shape, fortran_order, dtype = read_array_header_1_0(f)
        else:
            shape, fortran_order, dtype = read_array_header_2_0(f)
    return {
        "file": path.name,
        "shape": shape,
        "dtype": str(dtype),
        "fortran_order": fortran_order,
    }


def split_file_table(component_dir: Path, split: str):
    split_dir = require_path(component_dir / split)
    rows = []
    for path in sorted(split_dir.iterdir()):
        if path.suffix == ".npy":
            rows.append(npy_info(path))
        elif path.suffix == ".parquet":
            rows.append(parquet_info(path))
        else:
            rows.append({"file": path.name})
    return info_table(rows)


def graph_part_table(component_dir: Path, split: str):
    split_dir = require_path(component_dir / split)
    rows = []
    for path in sorted(split_dir.glob("graphs_part_*.pt")):
        rows.append(
            {
                "file": path.name,
                "size_mb": round(path.stat().st_size / 1024**2, 2),
            }
        )
    return info_table(rows)

## Gold Core

The core is the shared benchmark contract. It defines the train/test snapshot timestamps, prediction horizon, temporal split, and evaluation tables used to compare all models on the same target instances.

In [ ]:
core_dir = components["core"]
core_spec = read_yaml(require_path(core_dir / "dataset_core_spec.yaml"))
core_metadata = read_yaml(require_path(core_dir / "metadata.yaml"))

info_table(
    [
        {"field": "train period", "value": f"{core_spec['start_train_day']} to {core_spec['end_train_day']}"},
        {"field": "test period", "value": f"{core_spec['start_test_day']} to {core_spec['end_test_day']}"},
        {"field": "train snapshots", "value": len(core_spec["train_snapshots"])},
        {"field": "test snapshots", "value": len(core_spec["test_snapshots"])},
        {"field": "future events", "value": core_spec["n_future"]},
        {"field": "seed", "value": core_spec["seed"]},
    ]
)

`dataset_core_spec.yaml` stores the actual snapshot lists. The first few timestamps make the forward-in-time split explicit.

In [ ]:
pd.DataFrame(
    {
        "train_snapshots": pd.Series(core_spec["train_snapshots"][:5]),
        "test_snapshots": pd.Series(core_spec["test_snapshots"][:5]),
    }
)

`test_eval_table.parquet` is the shared evaluation table. Each row corresponds to one active train at one test snapshot, and its `future_delay_*` columns are the ground-truth delays at that train's future events. Model predictions are evaluated against these rows, which is what makes the benchmark comparison common across model families.

In [ ]:
test_eval_path = core_dir / "test_eval_table.parquet"
info_table([parquet_info(test_eval_path)])

In [ ]:
parquet_head(test_eval_path)

## Model-Specific Datasets

The four model-specific datasets are built on top of the same core. They differ in representation, not in the benchmark instances they are meant to evaluate.

In [ ]:
model_specific_summary = pd.DataFrame(
    [
        {
            "dataset": "tabular",
            "sample unit": "active train at one snapshot",
            "format": "NumPy arrays",
            "used by": "MLP, XGBoost, Transformer",
        },
        {
            "dataset": "sequential",
            "sample unit": "active train at one snapshot",
            "format": "NumPy arrays with static, past-event, and future-event tensors",
            "used by": "LSTM",
        },
        {
            "dataset": "gnn",
            "sample unit": "heterogeneous graph snapshot",
            "format": "PyTorch Geometric graph chunks",
            "used by": "GNN",
        },
        {
            "dataset": "graph_event",
            "sample unit": "test journeys/events plus railway graph and training travel-time statistics",
            "format": "Parquet tables and pickle statistics",
            "used by": "Graph-event baseline",
        },
    ]
)

with pd.option_context("display.max_colwidth", None):
    display(model_specific_summary)

## Tabular Dataset

The tabular dataset stores one fixed-size feature vector per active train at a snapshot. The `x`, `y`, and `y_mask` arrays hold features, future-delay targets, and target-validity masks; `scheme.yaml` names the columns.

In [ ]:
tabular_dir = components["tabular"]
tabular_scheme = read_yaml(require_path(tabular_dir / "scheme.yaml"))

split_file_table(tabular_dir, "train")

In [ ]:
info_table(
    [
        {"array": "x", "columns": len(tabular_scheme["x_columns"]), "first_columns": ", ".join(tabular_scheme["x_columns"][:8])},
        {"array": "y", "columns": len(tabular_scheme["y_columns"]), "first_columns": ", ".join(tabular_scheme["y_columns"][:8])},
        {"array": "y_mask", "columns": len(tabular_scheme["y_mask_columns"]), "first_columns": ", ".join(tabular_scheme["y_mask_columns"][:8])},
        {"array": "md", "columns": len(tabular_scheme["md_columns"]), "first_columns": ", ".join(tabular_scheme["md_columns"][:8])},
    ]
)

For one row index, `md` identifies the sample, `y` stores the future-delay targets, and `y_mask` marks which future targets are valid.

In [ ]:
row_idx = 0
tabular_md = np.load(tabular_dir / "train" / "md.npy", allow_pickle=True)
tabular_y = np.load(tabular_dir / "train" / "y.npy")
tabular_y_mask = np.load(tabular_dir / "train" / "y_mask.npy")

pd.DataFrame(
    [
        {
            "md": dict(zip(tabular_scheme["md_columns"], tabular_md[row_idx])),
            "y": tabular_y[row_idx].tolist(),
            "y_mask": tabular_y_mask[row_idx].tolist(),
        }
    ]
)

## Sequential Dataset

The sequential dataset keeps the same prediction target but separates inputs into static features, a past-event sequence, and a future-known-event sequence.

In [ ]:
sequential_dir = components["sequential"]
sequential_scheme = read_yaml(require_path(sequential_dir / "scheme.yaml"))

split_file_table(sequential_dir, "train")

In [ ]:
pd.DataFrame(
    [
        {"field": "static feature columns", "value": len(sequential_scheme["x_static_columns"])},
        {"field": "past-event step columns", "value": len(sequential_scheme["x_past_step_columns"])},
        {"field": "future-event step columns", "value": len(sequential_scheme["x_future_step_columns"])},
        {"field": "past slots", "value": sequential_scheme["past_slots"]},
        {"field": "future slots", "value": sequential_scheme["future_slots"]},
    ]
)

The sequential arrays follow the same row alignment as the tabular arrays: `md`, `y`, and `y_mask` share the same row index, while the input is split across static, past-sequence, and future-known-sequence tensors.

In [ ]:
row_idx = 0
sequential_md = np.load(sequential_dir / "train" / "md.npy", allow_pickle=True)
sequential_y = np.load(sequential_dir / "train" / "y.npy")
sequential_y_mask = np.load(sequential_dir / "train" / "y_mask.npy")

pd.DataFrame(
    [
        {
            "md": dict(zip(sequential_scheme["md_columns"], sequential_md[row_idx])),
            "y": sequential_y[row_idx].tolist(),
            "y_mask": sequential_y_mask[row_idx].tolist(),
        }
    ]
)

## GNN Dataset

The GNN dataset stores processed PyTorch Geometric graph chunks. Each `graphs_part_*.pt` file contains a list of heterogeneous graph snapshots with train nodes, station nodes, station-to-station railway edges, and train-to-station event edges. Reverse edge types are included in the graph objects, but the summary tables below show only the main forward edge types for readability. Loading these files requires `torch` and `torch_geometric`.

In [ ]:
gnn_dir = components["gnn"]

graph_part_table(gnn_dir, "train")

`feature_spec.yaml` records which feature columns were used when converting the intermediate node and edge tables into graph tensors.

In [ ]:
feature_spec = read_yaml(require_path(gnn_dir / "feature_spec.yaml"))

pd.DataFrame(
    [
        {"feature group": name, "n_features": len(columns), "first_features": ", ".join(columns[:8])}
        for name, columns in feature_spec.items()
    ]
)

Load one graph from the first chunk to inspect the object-level structure. The future train-to-station edge store contains both the edge features and the target tensor `y`.

In [ ]:
first_graph_part = sorted(require_path(gnn_dir / "train").glob("graphs_part_*.pt"))[1]
graphs = torch.load(first_graph_part, map_location="cpu", weights_only=False)
graph = graphs[0]

pd.DataFrame(
    [
        {"field": "snapshot_ts", "value": graph.snapshot_ts},
        {"field": "train nodes", "value": graph["train"].x.shape[0]},
        {"field": "station nodes", "value": graph["station"].x.shape[0]},
        {"field": "station-to-station edges", "value": graph[("station", "to", "station")].edge_index.shape[1]},
        {"field": "past train-to-station edges", "value": graph[("train", "past", "station")].edge_index.shape[1]},
        {"field": "future train-to-station edges", "value": graph[("train", "future", "station")].edge_index.shape[1]},
        {"field": "future targets", "value": graph[("train", "future", "station")].y.shape[0]},
    ]
)

## Graph-Event Dataset

The graph-event dataset supports the non-learning graph-event baseline. The `test/` directory contains the journeys and events needed at evaluation time; `node_links.parquet` and `travel_time_samples.pkl` live at the component root.

In [ ]:
graph_event_dir = components["graph_event"]

info_table(
    [
        {"path": "test/journeys.parquet", **parquet_info(graph_event_dir / "test" / "journeys.parquet")},
        {"path": "test/events.parquet", **parquet_info(graph_event_dir / "test" / "events.parquet")},
        {"path": "node_links.parquet", **parquet_info(graph_event_dir / "node_links.parquet")},
        {"path": "travel_time_samples.pkl", "exists": (graph_event_dir / "travel_time_samples.pkl").exists()},
    ]
)

`test/journeys.parquet` contains the test journeys used by the graph-event baseline.

In [ ]:
parquet_head(graph_event_dir / "test" / "journeys.parquet")

`test/events.parquet` contains the event rows associated with those test journeys.

In [ ]:
parquet_head(graph_event_dir / "test" / "events.parquet")

`node_links.parquet` is the railway graph used by the graph-event baseline.

In [ ]:
parquet_head(graph_event_dir / "node_links.parquet")

`travel_time_samples.pkl` stores training-derived travel-time samples used by the graph-event baseline. The cell below shows one actual entry.

In [ ]:
with require_path(graph_event_dir / "travel_time_samples.pkl").open("rb") as f:
    travel_time_samples = pickle.load(f)

example_key = next(iter(travel_time_samples))
example_value = travel_time_samples[example_key]

travel_time_key_columns = [
    "from_op_id",
    "from_event_type",
    "from_dep_line",
    "to_op_id",
    "to_event_type",
    "to_arr_line",
    "train_relation",
]
key_column = f"key ({', '.join(travel_time_key_columns)})"
row = {key_column: example_key}

if isinstance(example_value, dict):
    row.update(example_value)
elif isinstance(example_value, (list, tuple, np.ndarray, pd.Series)):
    row.update({f"value_{idx}": value for idx, value in enumerate(example_value[:10])})
else:
    row["value"] = example_value

pd.DataFrame([row])

[Back to README](../README.md#what-do-you-want-to-do)